# SERS Signal-Concentration Relationship Modeling

## Overview
This notebook demonstrates modeling the relationship between SERS signal intensity and Salmonella concentration using two simple global metrics:
1. **Max Peak Value** – maximum amplitude of the signal
2. **Full AUC** – area under the entire signal curve (trapezoidal integration)

## Key Considerations
- **Sensor Variability**: Signals from different sensors vary significantly in baseline/scaling
- **Per-Sensor Analysis**: We analyze each sensor individually rather than pooling all sensors
- **Exploratory Approach**: Focus on clarity, interpretability, and clean visualizations
- **New Naming Scheme**: Files are now named as `sensor_id + test_id + serotype` (e.g., `modelC-1_Test1_ST.xlsx`)

## Objectives
1. Load SERS data and inspect dataset structure
2. Compute max peak and full AUC metrics for each signal
3. Visualize signal vs. concentration relationships per sensor
4. Fit and compare regression models (linear, log-linear, logistic) for each sensor


In [ ]:
# Imports and Setup
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import integrate
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
import warnings

from sensd_sers_analysis.data import load_dataset_by_serotypes
from sensd_sers_analysis.utils import natural_sort, natural_sort_key

warnings.filterwarnings("ignore")

# Set plotting style
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ Imports and natural sorting functions completed successfully")

## 1. Data Loading and Inspection

Load SERS data and examine the dataset structure to understand:
- Signal arrays and their dimensions
- Concentration values and ranges
- Sensor IDs and their distribution
- Data quality and completeness


In [ ]:
# Load SERS Data
# --------------------------------------------------------------------------------------------------

# Define the root folder containing SERS dataset
data_folder = "../example_data/SERS Data 8 (April 2025)/"
signals_folders = ["March testing-Dilutions"]
serotypes = ["ST", "SE", "Mix"]

try:
    # Load the dataset using the new function
    data_df = load_dataset_by_serotypes(data_folder, signals_folders, serotypes)
    print(f"✅ Successfully loaded {len(data_df)} data entries")

except Exception as e:
    print(f"❌ Error loading data: {e}")
    data_df = pd.DataFrame()

In [ ]:
# Inspect Dataset Structure and Unique Identification
# --------------------------------------------------------------------------------------------------

if not data_df.empty:
    print("📊 Dataset Structure Analysis:")
    print("=" * 50)

    # Basic information
    print(f"Number of data entries: {len(data_df)}")
    print(f"Raman shift points: {len(data_df['raman_shift'].iloc[0])}")
    print(
        f"Raman shift range: {data_df['raman_shift'].iloc[0][0]:.1f} - "
        f"{data_df['raman_shift'].iloc[0][-1]:.1f} cm⁻¹"
    )

    # Signal dimensions
    print(f"DataFrame shape: {data_df.shape}")
    print(f"Columns: {list(data_df.columns)}")

    # Concentration information
    concentrations = data_df["concentration"].unique()
    sorted_conc = sorted(concentrations)
    print(f"Concentrations: {sorted_conc}")
    print(f"Concentration range: {min(sorted_conc):.1f} - {max(sorted_conc):.1f} CFU/ml")

    # Sensor and serotype distribution
    sensor_ids = data_df["sensor_id"].unique()
    serotypes_arr = data_df["serotype"].unique()

    # Fix: Ensure natural_sort only receives python list of strings/ints
    sensor_id_list = [str(sid) for sid in sensor_ids.tolist()]
    print(f"\nSensor IDs: {natural_sort(sensor_id_list)}")
    print(f"Serotypes: {set(serotypes_arr.tolist())}")
    print(f"Number of unique sensors: {len(sensor_ids)}")

    # Note: sensor_id already contains sensor configuration information (e.g., modelC-1)
    # No need to extract separate sensor_config since the naming scheme is now sensor_id + test_id + serotype

    # Count entries per sensor
    sensor_counts = data_df.groupby("sensor_id").size().reset_index(name="count")
    print("\nEntries per sensor:")
    # Fix: supply a Series of strings to the sort key
    sensor_counts_sorted = sensor_counts.sort_values(
        "sensor_id", key=lambda x: x.map(natural_sort_key)
    )
    for _, row in sensor_counts_sorted.iterrows():
        print(f"  Sensor {row['sensor_id']}: {row['count']} entries")

    # UNIQUE IDENTIFICATION ANALYSIS
    print("\n🔍 UNIQUE IDENTIFICATION ANALYSIS:")
    print("=" * 50)

    # Create composite unique identifier
    data_df["unique_id"] = (
        data_df["sensor_id"].astype(str)
        + "_"
        + data_df["test_id"].astype(str)
        + "_"
        + data_df["serotype"].astype(str)
        + "_"
        + data_df["signal_index"].astype(str)
    )

    # Analysis of uniqueness
    print(f"Total measurements: {len(data_df)}")
    print(
        f"Unique (sensor_id, test_id, signal_index): "
        f"{len(data_df[['sensor_id', 'test_id', 'signal_index']].drop_duplicates())}"
    )
    print(
        f"Unique (sensor_id, test_id, serotype, signal_index): "
        f"{len(data_df[['sensor_id', 'test_id', 'serotype', 'signal_index']].drop_duplicates())}"
    )
    print(f"Unique composite IDs: {len(data_df['unique_id'].unique())}")

    # Check for duplicates (core uniqueness)
    duplicates = data_df.groupby(["sensor_id", "test_id", "signal_index"]).size()
    duplicate_groups = duplicates[duplicates > 1]

    if not duplicate_groups.empty:
        print(f"⚠️ Found {len(duplicate_groups)} groups with multiple measurements:")
        for (sensor_id, test_id, signal_idx), count in duplicate_groups.head().items():
            print(
                f"  Sensor {sensor_id}, Test {test_id}, Signal {signal_idx}: {count} measurements"
            )
    else:
        print("✅ No duplicate measurements found - each combination is unique")

    # Show unique identification components
    print("\n📋 UNIQUE IDENTIFICATION COMPONENTS:")
    print("Each measurement is uniquely identified by:")
    print("  • Sensor ID: Physical sensor unit identifier (includes config info, e.g., modelC-1)")
    print("  • Test ID: Specific test/experiment identifier (e.g., Test1, Test2)")
    print("  • Serotype: Pathogen type (ST/SE) - included for informational purposes")
    print("  • Signal Index: Position within the measurement file (0, 1, 2, 3, ...)\n")
    print("Note: Core uniqueness is determined by (sensor_id, test_id, signal_index)")
    print("Serotype is included for convenience and information, but not required for uniqueness.")
    print("\nAdditional metadata:")
    print(
        "  • Concentration: CFU/ml value\n"
        "  • Sensor ID: Contains sensor configuration info (e.g., modelC-1 indicates modelC configuration)"
    )

    # Display sample with unique IDs
    print("\n📋 Sample of data with unique identifiers:")
    sample_cols = [
        "unique_id",
        "sensor_id",
        "test_id",
        "signal_index",
        "serotype",
        "concentration",
    ]
    print(data_df[sample_cols].head(10))

else:
    print("❌ No data loaded - cannot proceed with analysis")

## 2. Metric Computation Functions

Implement functions to compute the two key metrics:
- **Max Peak Value**: Maximum amplitude of each signal
- **Full AUC**: Area under the entire signal curve using trapezoidal integration


In [ ]:
def compute_metrics(data_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute signal metrics for each signal in the dataset.

    This includes:
        - max_peak: Maximum amplitude of the full signal
        - full_auc: Area under the entire signal curve
        - max_peak_pre700: Max amplitude where raman_shift < 700
        - auc_pre700: AUC where raman_shift < 700
        - max_peak_700_900: Max amplitude where 700 <= raman_shift < 900
        - auc_700_900: AUC where 700 <= raman_shift < 900
        - max_peak_900_1200: Max amplitude where 900 <= raman_shift < 1200
        - auc_900_1200: AUC where 900 <= raman_shift < 1200

    Raman shift units are assumed to be in wavenumbers (cm⁻¹).

    Args:
        data_df (pd.DataFrame): DataFrame from load_dataset_by_serotypes

    Returns:
        pd.DataFrame: DataFrame with identification and computed metrics
    """
    metrics_data: list[dict[str, float | str | int]] = []

    for _, row in data_df.iterrows():
        # Unique identification
        sensor_id = row["sensor_id"]
        test_id = row["test_id"]
        serotype = row["serotype"]
        concentration = row["concentration"]
        signal_index = row["signal_index"]
        filename = row["filename"]
        unique_id = f"{sensor_id}_{test_id}_{serotype}_{signal_index}"

        # Signal data
        signal = row["signal"]  # np.ndarray
        raman_shift = row["raman_shift"]  # np.ndarray

        # Safety: arrays must match; validate shape
        if len(signal) != len(raman_shift):
            raise ValueError(
                f"Signal length and raman_shift length do not match for unique_id {unique_id}: "
                f"{len(signal)} vs {len(raman_shift)}"
            )

        # Compute global max and AUC
        max_peak: float = np.max(signal)
        full_auc: float = integrate.trapezoid(signal, raman_shift)

        # Region masks
        mask_pre700 = raman_shift < 700
        mask_700_900 = (raman_shift >= 700) & (raman_shift < 900)
        mask_900_1200 = (raman_shift >= 900) & (raman_shift < 1200)

        # Pre-700 cm⁻¹ region
        if np.any(mask_pre700):
            max_peak_pre700: float = np.max(signal[mask_pre700])
            auc_pre700: float = integrate.trapezoid(signal[mask_pre700], raman_shift[mask_pre700])
        else:
            max_peak_pre700 = np.nan
            auc_pre700 = np.nan

        # 700–900 cm⁻¹ region
        if np.any(mask_700_900):
            max_peak_700_900: float = np.max(signal[mask_700_900])
            auc_700_900: float = integrate.trapezoid(
                signal[mask_700_900], raman_shift[mask_700_900]
            )
        else:
            max_peak_700_900 = np.nan
            auc_700_900 = np.nan

        # 900–1200 cm⁻¹ region
        if np.any(mask_900_1200):
            max_peak_900_1200: float = np.max(signal[mask_900_1200])
            auc_900_1200: float = integrate.trapezoid(
                signal[mask_900_1200], raman_shift[mask_900_1200]
            )
        else:
            max_peak_900_1200 = np.nan
            auc_900_1200 = np.nan

        metrics_data.append(
            {
                # Unique identification
                "unique_id": unique_id,
                "sensor_id": sensor_id,
                "test_id": test_id,
                "serotype": serotype,
                "concentration": concentration,
                "signal_index": signal_index,
                "filename": filename,
                # Computed metrics
                "log_concentration": row["log_concentration"],
                "max_peak": max_peak,
                "full_auc": full_auc,
                "max_peak_pre700": max_peak_pre700,
                "auc_pre700": auc_pre700,
                "max_peak_700_900": max_peak_700_900,
                "auc_700_900": auc_700_900,
                "max_peak_900_1200": max_peak_900_1200,
                "auc_900_1200": auc_900_1200,
            }
        )

    return pd.DataFrame(metrics_data)


# Compute metrics for all data
if not data_df.empty:
    print("🔄 Computing signal metrics...")
    metrics_df = compute_metrics(data_df)

    print(f"✅ Computed metrics for {len(metrics_df)} signal measurements")
    print(f"📊 Metrics DataFrame shape: {metrics_df.shape}")
    print(f"📊 Columns: {list(metrics_df.columns)}")

    # Display sample of the data
    print("\n📋 Sample of computed metrics:")
    print(metrics_df.head(10))

else:
    print("❌ No data available for metric computation")
    metrics_df = pd.DataFrame()

## 2.5. Unique Identification Analysis

Demonstrate how to use the unique identification system for data analysis and quality control.


In [ ]:
# Unique Identification Analysis
# --------------------------------------------------------------------------------------------------

if not data_df.empty:
    print("🔍 UNIQUE IDENTIFICATION ANALYSIS")
    print("=" * 60)

    # 1. Analyze measurement distribution by sensor (natural sorted)
    print("\n1. MEASUREMENT DISTRIBUTION BY SENSOR:")

    sensor_analysis = (
        data_df.groupby("sensor_id")
        .agg(
            {
                "test_id": "nunique",
                "serotype": "nunique",
                "concentration": "nunique",
                "unique_id": "count",
            }
        )
        .rename(
            columns={
                "test_id": "n_tests",
                "serotype": "n_serotypes",
                "concentration": "n_concentrations",
                "unique_id": "n_measurements",
            }
        )
    )
    # Sort the DataFrame index using natural sort order
    natural_sorted_index = natural_sort(list(sensor_analysis.index))
    sensor_analysis = sensor_analysis.loc[natural_sorted_index]

    print(sensor_analysis.head(10))

    # 2. Analyze test structure (apply natural sort by both sensor_id and test_id)
    print("\n2. TEST STRUCTURE ANALYSIS:")
    test_analysis = (
        data_df.groupby(["sensor_id", "test_id"])
        .agg({"serotype": "first", "concentration": ["count", "nunique"], "unique_id": "count"})
        .round(2)
    )
    test_analysis.columns = ["serotype", "n_measurements", "n_concentrations", "total_measurements"]

    # Apply natural sort to MultiIndex (sensor_id, test_id)
    natural_sorted_sensor_ids = natural_sort([idx[0] for idx in test_analysis.index])

    # For each sensor, get all its associated test_ids, and apply natural sort
    def multiindex_natural_sort(idx):
        # idx: pandas MultiIndex
        indexed = list(idx)

        # Create sort key as tuple: (sensor_natural_sort_key, test_natural_sort_key)
        def sort_key(item):
            # item is (sensor_id, test_id)
            return (
                natural_sort_key(str(item[0])),
                natural_sort_key(str(item[1])),
            )

        sorted_idx = sorted(indexed, key=sort_key)
        return sorted_idx

    natural_sorted_test_index = multiindex_natural_sort(test_analysis.index)
    test_analysis = test_analysis.loc[natural_sorted_test_index]
    print(test_analysis.head(10))

    # 3. Check for potential quality issues
    print("\n3. QUALITY CONTROL CHECKS:")

    # Check for tests with multiple serotypes
    multi_serotype_tests = data_df.groupby(["sensor_id", "test_id"])["serotype"].nunique()
    multi_serotype_tests = multi_serotype_tests[multi_serotype_tests > 1]
    if len(multi_serotype_tests) > 0:
        print(f"⚠️ Tests with multiple serotypes: {len(multi_serotype_tests)} tests")
        print("   This might indicate mixed samples or data quality issues")
    else:
        print("✅ All tests use single serotype (as expected)")

    # 4. Show examples of unique identification (naturally sort sensor display)
    print("\n4. EXAMPLES OF UNIQUE IDENTIFICATION:")
    print("Each measurement can be uniquely identified using:")
    example_sensors = data_df["sensor_id"].drop_duplicates()
    example_sensors = example_sensors.loc[
        example_sensors.map(lambda s: s in natural_sorted_index)
    ].head(3)

    for sensor_id in example_sensors:
        sensor_data = data_df[data_df["sensor_id"] == sensor_id].head(3)
        print(f"\nSensor {sensor_id} examples:")
        for _, row in sensor_data.iterrows():
            print(
                f"  • {row['unique_id']} (Test {row['test_id']}, Signal {row['signal_index']}, {row['serotype']}, {row['concentration']} CFU/ml)"
            )

    # 5. Demonstrate filtering by unique criteria
    print("\n5. FILTERING EXAMPLES:")

    # Filter by specific sensor and test
    sensor_1_test_1 = data_df[
        (data_df["sensor_id"] == "modelC-1") & (data_df["test_id"] == "Test1")
    ]
    print(f"Sensor modelC-1, Test Test1: {len(sensor_1_test_1)} measurements")
    if len(sensor_1_test_1) > 0:
        # natural sort for concentrations
        nat_conc = sorted(sensor_1_test_1["concentration"].unique(), key=lambda x: float(x))
        print(f"  Concentrations: {nat_conc}")
        print(f"  Serotype: {sensor_1_test_1['serotype'].iloc[0]}")

    # Filter by concentration range (sensor IDs naturally sorted)
    high_conc = data_df[data_df["concentration"] >= 100]
    nat_high_conc_sensors = natural_sort(list(high_conc["sensor_id"].unique()))
    print(f"High concentration (≥100 CFU/ml): {len(high_conc)} measurements")
    print(f"  Sensor IDs: {nat_high_conc_sensors}")

else:
    print("❌ No data available for analysis")

## 3. Visualization Functions

Create functions to visualize signal-concentration relationships for each sensor individually, as sensor variability requires separate analysis.


In [ ]:
def plot_peak_relationships_separate(metrics_df: pd.DataFrame, sensor_ids: list = None) -> None:
    """
    Plot peak signal-concentration relationships in separate plots for each sensor.
    Each test within a sensor gets a unique color and connected points.
    Plots four peak signal metrics vs concentration for each sensor (in a 1x4 horizontal grid):
        - max_peak
        - max_peak_pre700
        - max_peak_700_900
        - max_peak_900_1200

    Sensors are ordered numerically by their ID number.

    Args:
        metrics_df (pd.DataFrame): DataFrame with computed metrics.
        sensor_ids (list, optional): List of sensor IDs to plot. If None, plot all sensors.

    Returns:
        None

    Notes
    -----
    - Units: Concentration in CFU/ml, Max Peak values in arbitrary intensity units.
    - Each subplot visualizes a different Raman peak summary for SERS signal intensity.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import re

    if metrics_df.empty:
        print("❌ No data available for plotting")
        return

    # Only peak metrics, pretty names/ylabels and marker style
    plot_metrics = [
        ("max_peak", "Max Peak Value", "o"),
        ("max_peak_pre700", "Max Peak (<700 cm⁻¹)", "D"),
        ("max_peak_700_900", "Max Peak (700–900 cm⁻¹)", "^"),
        ("max_peak_900_1200", "Max Peak (900–1200 cm⁻¹)", "P"),
    ]

    n_rows, n_cols = 1, 4

    # Get unique sensor IDs and sort numerically by sensor number
    if sensor_ids is None:
        all_sensor_ids = metrics_df["sensor_id"].unique()
    else:
        all_sensor_ids = sensor_ids

    def extract_sensor_number(sensor_id: str) -> int:
        """Extract numeric part from sensor ID (e.g., 'modelC-15' -> 15)"""
        match = re.search(r"(\d+)", sensor_id)
        return int(match.group(1)) if match else 0

    sensor_ids_sorted = sorted(all_sensor_ids, key=extract_sensor_number)

    # Define color palette for tests
    test_colors = sns.color_palette("husl", 20)  # Support up to 20 different tests

    for sensor_id in sensor_ids_sorted:
        sensor_data = metrics_df[metrics_df["sensor_id"] == sensor_id]

        if len(sensor_data) == 0:
            continue

        # Make a wider, less tall figure (e.g., 15in x 3.5in)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5), constrained_layout=True)
        axes = axes.flatten() if n_cols > 1 else [axes]

        suptitle = f"Sensor {sensor_id} - Peak Metrics vs Concentration"
        fig.suptitle(suptitle, fontsize=16, fontweight="bold", y=1.04)

        unique_tests = sorted(sensor_data["test_id"].unique())

        # Plot each peak metric as a subplot
        for plot_idx, (metric, ylabel, marker) in enumerate(plot_metrics):
            ax = axes[plot_idx]

            for test_idx, test_id in enumerate(unique_tests):
                test_data = sensor_data[sensor_data["test_id"] == test_id]
                color = test_colors[test_idx % len(test_colors)]

                # Sort by concentration for line connection
                test_data_sorted = test_data.sort_values("concentration")

                ax.scatter(
                    test_data_sorted["concentration"],
                    test_data_sorted[metric],
                    alpha=0.8,
                    label=f"{test_id}" if plot_idx == 0 else None,  # Label only first subplot
                    color=color,
                    s=48,
                    marker=marker,
                )

                # Connect points with lines for each test
                if len(test_data_sorted) > 1:
                    ax.plot(
                        test_data_sorted["concentration"],
                        test_data_sorted[metric],
                        color=color,
                        alpha=0.7,
                        linewidth=1.8,
                        linestyle="-",
                    )

            ax.set_xlabel("Concentration (CFU/ml)")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{ylabel} vs Concentration", fontsize=11)
            ax.grid(True, alpha=0.25)
            # Log scale for wide concentration range
            concentrations = sensor_data["concentration"]
            if (
                concentrations.max() > 0
                and concentrations.min() > 0
                and concentrations.max() / concentrations.min() > 10
            ):
                ax.set_xscale("log")
            # Only add legend to first plot for clarity
            if plot_idx == 0:
                ax.legend(bbox_to_anchor=(1.03, 1), loc="upper left", fontsize=7, title="Test ID")

        # Hide unused subplots if any (should not happen for 1x4)
        for idx in range(len(plot_metrics), len(axes)):
            axes[idx].set_visible(False)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()


# Create visualizations
if not metrics_df.empty:
    print("📊 Creating peak-only signal-concentration relationship plots...")

    # Get sensor IDs with sufficient data
    sensor_id_counts = metrics_df["sensor_id"].value_counts()
    sorted_sensor_ids = natural_sort(metrics_df["sensor_id"].unique().tolist())
    sensor_counts = sensor_id_counts.reindex(sorted_sensor_ids).dropna().astype(int)
    valid_sensors = sensor_counts[sensor_counts >= 3].index.tolist()  # At least 3 data points

    if valid_sensors:
        print(f"📈 Plotting relationships for sensors: {valid_sensors}")
        plot_peak_relationships_separate(metrics_df, valid_sensors)
    else:
        print("⚠️ No sensors with sufficient data points for plotting")
else:
    print("❌ No metrics data available for visualization")

## 4. Regression Model Functions

Implement functions to fit and compare different regression models for each sensor:
- **Linear model**: y = a * x + b
- **Log-linear model**: y = a * log(x) + b  
- **Polynomial model**: y = a * x² + b * x + c
- **Logistic model**: y = a / (1 + exp(-b * (x - c))) (for saturation effects)


In [ ]:
def fit_regression_models(
    metrics_df: pd.DataFrame, sensor_id: str, metric: str = "max_peak"
) -> dict:
    """
    Fit multiple regression models for a specific sensor and metric.

    Args:
        metrics_df: DataFrame with computed metrics
        sensor_id: Sensor ID to analyze
        metric: Metric to model ('max_peak' or 'full_auc')

    Returns:
        dict: Dictionary containing fitted models and their performance metrics
    """
    # Filter data for specific sensor
    sensor_data = metrics_df[metrics_df["sensor_id"] == sensor_id].copy()

    if len(sensor_data) < 3:
        return {"error": f"Insufficient data points for sensor {sensor_id}"}

    # Prepare data
    x = sensor_data["concentration"].values
    y = sensor_data[metric].values

    # Remove any invalid values
    valid_mask = (x > 0) & np.isfinite(y)
    x = x[valid_mask]
    y = y[valid_mask]

    if len(x) < 3:
        return {"error": f"Insufficient valid data points for sensor {sensor_id}"}

    models = {}

    # 1. Linear Model: y = a * x + b
    try:
        linear_model = LinearRegression()
        linear_model.fit(x.reshape(-1, 1), y)
        y_pred_linear = linear_model.predict(x.reshape(-1, 1))
        r2_linear = r2_score(y, y_pred_linear)

        models["linear"] = {
            "model": linear_model,
            "params": {"slope": linear_model.coef_[0], "intercept": linear_model.intercept_},
            "r2": r2_linear,
            "predictions": y_pred_linear,
        }
    except Exception as e:
        models["linear"] = {"error": str(e)}

    # 2. Log-Linear Model: y = a * log(x) + b
    try:
        log_x = np.log10(x)
        log_linear_model = LinearRegression()
        log_linear_model.fit(log_x.reshape(-1, 1), y)
        y_pred_log_linear = log_linear_model.predict(log_x.reshape(-1, 1))
        r2_log_linear = r2_score(y, y_pred_log_linear)

        models["log_linear"] = {
            "model": log_linear_model,
            "params": {
                "slope": log_linear_model.coef_[0],
                "intercept": log_linear_model.intercept_,
            },
            "r2": r2_log_linear,
            "predictions": y_pred_log_linear,
        }
    except Exception as e:
        models["log_linear"] = {"error": str(e)}

    # 3. Polynomial Model (degree 2): y = a * x² + b * x + c
    try:
        poly_features = PolynomialFeatures(degree=2)
        x_poly = poly_features.fit_transform(x.reshape(-1, 1))
        poly_model = LinearRegression()
        poly_model.fit(x_poly, y)
        y_pred_poly = poly_model.predict(x_poly)
        r2_poly = r2_score(y, y_pred_poly)

        models["polynomial"] = {
            "model": poly_model,
            "poly_features": poly_features,
            "params": {"coefs": poly_model.coef_, "intercept": poly_model.intercept_},
            "r2": r2_poly,
            "predictions": y_pred_poly,
        }
    except Exception as e:
        models["polynomial"] = {"error": str(e)}

    # 4. Logistic Model: y = a / (1 + exp(-b * (x - c)))
    try:

        def logistic_func(x, a, b, c):
            return a / (1 + np.exp(-b * (x - c)))

        # Initial parameter estimates
        a_init = np.max(y)  # Maximum value
        b_init = 1.0 / (np.max(x) - np.min(x))  # Steepness
        c_init = np.median(x)  # Midpoint

        popt, _ = curve_fit(logistic_func, x, y, p0=[a_init, b_init, c_init], maxfev=1000)
        y_pred_logistic = logistic_func(x, *popt)
        r2_logistic = r2_score(y, y_pred_logistic)

        models["logistic"] = {
            "params": {"a": popt[0], "b": popt[1], "c": popt[2]},
            "r2": r2_logistic,
            "predictions": y_pred_logistic,
            "function": logistic_func,
        }
    except Exception as e:
        models["logistic"] = {"error": str(e)}

    return {
        "sensor_id": sensor_id,
        "metric": metric,
        "n_points": len(x),
        "x_data": x,
        "y_data": y,
        "models": models,
    }


def plot_regression_results(results: dict) -> None:
    """
    Plot regression results for a sensor showing data points and fitted curves.

    Args:
        results: Results dictionary from fit_regression_models
    """
    if "error" in results:
        print(f"❌ {results['error']}")
        return

    sensor_id = results["sensor_id"]
    metric = results["metric"]
    x = results["x_data"]
    y = results["y_data"]
    models = results["models"]

    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    # Plot data points
    ax.scatter(x, y, alpha=0.7, s=60, color="black", label="Data points")

    # Plot fitted curves
    x_smooth = np.linspace(x.min(), x.max(), 100)

    # Linear model
    if "linear" in models and "error" not in models["linear"]:
        y_linear = models["linear"]["model"].predict(x_smooth.reshape(-1, 1))
        ax.plot(
            x_smooth,
            y_linear,
            "b-",
            linewidth=2,
            label=f"Linear (R² = {models['linear']['r2']:.3f})",
        )

    # Log-linear model
    if "log_linear" in models and "error" not in models["log_linear"]:
        log_x_smooth = np.log10(x_smooth)
        y_log_linear = models["log_linear"]["model"].predict(log_x_smooth.reshape(-1, 1))
        ax.plot(
            x_smooth,
            y_log_linear,
            "r-",
            linewidth=2,
            label=f"Log-linear (R² = {models['log_linear']['r2']:.3f})",
        )

    # Polynomial model
    if "polynomial" in models and "error" not in models["polynomial"]:
        poly_features = models["polynomial"]["poly_features"]
        x_poly_smooth = poly_features.transform(x_smooth.reshape(-1, 1))
        y_poly = models["polynomial"]["model"].predict(x_poly_smooth)
        ax.plot(
            x_smooth,
            y_poly,
            "g-",
            linewidth=2,
            label=f"Polynomial (R² = {models['polynomial']['r2']:.3f})",
        )

    # Logistic model
    if "logistic" in models and "error" not in models["logistic"]:
        logistic_func = models["logistic"]["function"]
        params = models["logistic"]["params"]
        y_logistic = logistic_func(x_smooth, params["a"], params["b"], params["c"])
        ax.plot(
            x_smooth,
            y_logistic,
            "m-",
            linewidth=2,
            label=f"Logistic (R² = {models['logistic']['r2']:.3f})",
        )

    # Formatting
    ax.set_xlabel("Concentration (CFU/ml)")
    ax.set_ylabel(f"{metric.replace('_', ' ').title()}")
    ax.set_title(f"Sensor {sensor_id} - {metric.replace('_', ' ').title()} vs Concentration")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Use log scale if concentration range is large
    if x.max() / x.min() > 10:
        ax.set_xscale("log")

    plt.tight_layout()
    plt.show()


print("✅ Regression model functions defined")

## 5. Complete Workflow Demonstration

Run the complete analysis workflow for selected sensors, demonstrating:
1. Metric computation
2. Visualization of relationships
3. Regression model fitting and comparison
4. Performance evaluation


In [ ]:
# Run Complete Analysis for Selected Sensors
# --------------------------------------------------------------------------------------------------

if not metrics_df.empty:
    # Select sensors with sufficient data
    sensor_counts = metrics_df["sensor_id"].value_counts()
    valid_sensors = sensor_counts[sensor_counts >= 3].index.tolist()

    print(f"📊 Analyzing {len(valid_sensors)} sensors with sufficient data")
    print("=" * 70)

    # Analyze first 3 sensors (or fewer if not available)
    sensors_to_analyze = valid_sensors[:3]

    for sensor_id in sensors_to_analyze:
        print(f"\n{'=' * 70}")
        print(f"🔬 SENSOR {sensor_id} ANALYSIS")
        print(f"{'=' * 70}")

        # Analyze both metrics
        for metric in ["max_peak_pre700", "full_auc"]:
            print(f"\n📈 Analyzing {metric.replace('_', ' ').title()}...")

            # Fit regression models
            results = fit_regression_models(metrics_df, sensor_id, metric)

            if "error" not in results:
                # Display model performance
                print("\nModel Performance (R² scores):")
                models = results["models"]

                for model_name, model_info in models.items():
                    if "error" not in model_info:
                        r2 = model_info["r2"]
                        print(f"  {model_name.capitalize():15s}: R² = {r2:.4f}")

                # Find best model
                best_model = max(
                    [(name, info["r2"]) for name, info in models.items() if "error" not in info],
                    key=lambda x: x[1],
                )
                print(f"\n✅ Best model: {best_model[0].capitalize()} (R² = {best_model[1]:.4f})")

                # Plot results
                plot_regression_results(results)
            else:
                print(f"❌ {results['error']}")

        print(f"\n{'=' * 70}\n")

else:
    print("❌ No metrics data available for analysis")

## 6. Summary Statistics and Model Comparison

Generate comprehensive summary statistics across all sensors to understand overall trends and model performance.


In [ ]:
# Generate Summary Statistics Across All Sensors
# --------------------------------------------------------------------------------------------------

if not metrics_df.empty:
    print("📊 SUMMARY STATISTICS")
    print("=" * 70)

    # Overall statistics
    print("\n1. Overall Metrics Summary:")
    print(metrics_df[["concentration", "max_peak", "full_auc"]].describe())

    # Per-sensor statistics
    print("\n2. Per-Sensor Statistics:")
    sensor_stats = (
        metrics_df.groupby("sensor_id")
        .agg(
            {
                "concentration": ["count", "min", "max"],
                "max_peak": ["mean", "std"],
                "full_auc": ["mean", "std"],
            }
        )
        .round(2)
    )
    print(sensor_stats)

    # Serotype comparison
    print("\n3. Serotype Comparison:")
    serotype_stats = (
        metrics_df.groupby("serotype")
        .agg({"max_peak": ["mean", "std"], "full_auc": ["mean", "std"]})
        .round(2)
    )
    print(serotype_stats)

    # Model performance summary across all sensors
    print("\n4. Model Performance Summary:")
    print("   Fitting regression models for all sensors...")

    all_results = []
    valid_sensors = metrics_df["sensor_id"].value_counts()[lambda x: x >= 3].index

    for sensor_id in valid_sensors:
        for metric in ["max_peak", "full_auc"]:
            results = fit_regression_models(metrics_df, sensor_id, metric)
            if "error" not in results:
                for model_name, model_info in results["models"].items():
                    if "error" not in model_info:
                        all_results.append(
                            {
                                "sensor_id": sensor_id,
                                "metric": metric,
                                "model": model_name,
                                "r2": model_info["r2"],
                            }
                        )

    results_df = pd.DataFrame(all_results)

    if not results_df.empty:
        # Average R² by model type
        model_performance = results_df.groupby("model")["r2"].agg(["mean", "std", "min", "max"])
        model_performance = model_performance.sort_values("mean", ascending=False)
        print("\n   Average R² scores by model type:")
        print(model_performance.round(4))

        # Best model for each sensor/metric combination
        best_models = results_df.loc[results_df.groupby(["sensor_id", "metric"])["r2"].idxmax()]
        print("\n   Best model distribution:")
        print(best_models["model"].value_counts())

else:
    print("❌ No metrics data available")